In [1]:
import numpy as np
import pandas as pd

import requests
import tarfile
import io
from pathlib import Path
import gzip

import os
import folium
import random

import pickle


In [2]:
# Load your enriched dataset if not already in memory
df_enriched = pd.read_parquet("lfbo/lfbo_2024_enriched.parquet")

# Find Mondays only because it's one of the busiest days and it's the only one freely accessible in the dataset
df_mondays = df_enriched[df_enriched["day_of_week"] == 0].copy()

print(f"Monday flights in 2024: {len(df_mondays)}")

Monday flights in 2024: 5884


In [3]:
# Toulouse TMA bounding box
LFBO_LAT_MIN, LFBO_LAT_MAX = 42.5, 44.5
LFBO_LON_MIN, LFBO_LON_MAX = 0.5,  2.5
BASE_URL = "https://s3.opensky-network.org/data-samples/states"


Path("data/raw/states").mkdir(parents=True, exist_ok=True)

def download_state_vectors_lfbo(date: str, hour: str) -> pd.DataFrame:
    filename = f"states_{date}-{hour}.csv.tar"
    path = Path(f"data/raw/states/{filename}")
    
    filtered_path = Path(f"data/raw/states/filtered_{date}-{hour}.parquet")
    if filtered_path.exists():
        print(f"Loading filtered cache: {filtered_path.name}")
        return pd.read_parquet(filtered_path)
    
    if not path.exists():
        url = f"{BASE_URL}/.{date}/{hour}/{filename}"
        print(f"Downloading {filename}...")
        r = requests.get(url, stream=True)
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
    
    with tarfile.open(path, "r") as tar:
        gz_file = [m for m in tar.getmembers() if m.name.endswith(".csv.gz")][0]
        f = tar.extractfile(gz_file)
        with gzip.open(f, "rb") as gz:
            df = pd.read_csv(gz)
    
    df = df[df["lat"].between(LFBO_LAT_MIN, LFBO_LAT_MAX) & df["lon"].between(LFBO_LON_MIN, LFBO_LON_MAX)].copy()
    df["time"] = pd.to_datetime(df["time"], unit="s", utc=True)
    
    df.to_parquet(filtered_path, index=False)
    print(f"Filtered {len(df)} rows saved to {filtered_path.name}")
    
    return df

df_lfbo = download_state_vectors_lfbo("2017-07-03", "07")
print(f"\nLFBO rows: {len(df_lfbo)}")
print(f"Unique aircraft: {df_lfbo['icao24'].nunique()}")

Loading filtered cache: filtered_2017-07-03-07.parquet

LFBO rows: 9359
Unique aircraft: 119


In [4]:
targets = [
    ("2017-07-03", ["06", "07", "08", "15", "16"]),  # summer
    ("2017-10-02", ["06", "07", "08", "15", "16"]),  # autumn
    ("2018-01-08", ["06", "07", "08", "15", "16"]),  # winter
]

all_states = []
for date, hours in targets:
    for hour in hours:
        df = download_state_vectors_lfbo(date, hour)
        df["source_date"] = date
        all_states.append(df)
        
states_raw = pd.concat(all_states, ignore_index=True)

Filtered 8803 rows saved to filtered_2017-07-03-06.parquet
Loading filtered cache: filtered_2017-07-03-07.parquet
Filtered 9808 rows saved to filtered_2017-07-03-08.parquet
Filtered 8899 rows saved to filtered_2017-07-03-15.parquet
Filtered 9489 rows saved to filtered_2017-07-03-16.parquet
Filtered 8524 rows saved to filtered_2017-10-02-06.parquet
Filtered 8165 rows saved to filtered_2017-10-02-07.parquet
Filtered 10093 rows saved to filtered_2017-10-02-08.parquet
Filtered 7288 rows saved to filtered_2017-10-02-15.parquet
Filtered 8961 rows saved to filtered_2017-10-02-16.parquet
Filtered 2585 rows saved to filtered_2018-01-08-06.parquet
Filtered 4372 rows saved to filtered_2018-01-08-07.parquet
Filtered 3622 rows saved to filtered_2018-01-08-08.parquet
Filtered 7308 rows saved to filtered_2018-01-08-15.parquet
Filtered 6002 rows saved to filtered_2018-01-08-16.parquet


In [5]:
print(f"\nTotal state vectors: {len(states_raw)}")
print(f"Unique aircraft: {states_raw['icao24'].nunique()}")
print(f"Date coverage: {states_raw['source_date'].value_counts()}")


Total state vectors: 113278
Unique aircraft: 957
Date coverage: source_date
2017-07-03    46358
2017-10-02    43031
2018-01-08    23889
Name: count, dtype: int64


In [6]:
from src.track_reconstruction import reconstruct_tracks, reconstruction_report

tracks = reconstruct_tracks(states_raw)
reconstruction_report(tracks)

Total tracks reconstructed: 1332

Duration (minutes):
  mean=14.0  median=14.0  max=179.0

Point count per track:
  mean=85.2  median=85.0  max=1075.0

Tracks with gaps: 0 (0.0%)


In [7]:
LFBO_LAT, LFBO_LON = 43.6293, 1.3673
ARRIVAL_RADIUS_DEG = 0.15  # ~15km radius around LFBO

def is_arriving_lfbo(track: "TrackSegment") -> bool:
    last_point = track.points.dropna(subset=["lat", "lon"]).iloc[-1]
    dist = np.sqrt(
        (last_point["lat"] - LFBO_LAT)**2 + (last_point["lon"] - LFBO_LON)**2)
    
    return dist < ARRIVAL_RADIUS_DEG

def is_departing_lfbo(track: "TrackSegment") -> bool:
    first_point = track.points.dropna(subset=["lat", "lon"]).iloc[0]
    dist = np.sqrt((first_point["lat"] - LFBO_LAT)**2 + (first_point["lon"] - LFBO_LON)**2)
    return dist < ARRIVAL_RADIUS_DEG

for track in tracks:
    track.is_arriving = is_arriving_lfbo(track)
    track.is_departing = is_departing_lfbo(track)
    track.is_transiting = not track.is_arriving and not track.is_departing

arriving  = [t for t in tracks if t.is_arriving]
departing = [t for t in tracks if t.is_departing]
transiting = [t for t in tracks if t.is_transiting]

print(f"Arriving:  {len(arriving)}")
print(f"Departing: {len(departing)}")
print(f"Transiting:{len(transiting)}")

Arriving:  126
Departing: 156
Transiting:1086


In [ ]:
def plot_tracks_classified(tracks: list, n: int = 5, output_path: str = "figures/track_samples.html") -> None:
    arriving   = [t for t in tracks if t.is_arriving]
    departing  = [t for t in tracks if t.is_departing]
    transiting = [t for t in tracks if t.is_transiting]
    
    n_each = max(1, n // 3)
    sample = (
        random.sample(arriving,   min(n_each, len(arriving)))   +
        random.sample(departing,  min(n_each, len(departing)))  +
        random.sample(transiting, min(n_each, len(transiting))))
    
    m = folium.Map(location=[LFBO_LAT, LFBO_LON], zoom_start=10)
    
    folium.Marker(
        [LFBO_LAT, LFBO_LON],
        popup="LFBO Toulouse-Blagnac",
        icon=folium.Icon(color="red", icon="plane")).add_to(m)
    
    color_map = {
        "arriving":  "blue",
        "departing": "green",
        "transiting":"gray"}
    
    for track in sample:
        points = track.points[["lat", "lon"]].dropna().values.tolist()
        if len(points) < 2:
            continue
        
        category = (
            "arriving"  if track.is_arriving  else
            "departing" if track.is_departing else
            "transiting")
        
        folium.PolyLine(
            points,
            color=color_map[category],
            weight=2,
            opacity=0.8,
            popup=(
                f"icao24: {track.icao24}<br>"
                f"callsign: {track.callsign}<br>"
                f"category: {category}<br>"
                f"duration: {track.duration_seconds/60:.1f} min<br>"
                f"date: {track.source_date}"
            )).add_to(m)
    
    legend_html = """
    <div style="position:fixed; bottom:30px; left:30px; z-index:1000; 
                background:white; padding:10px; border:1px solid grey">
        <b>Track Category</b><br>
        <span style="color:blue">&#9644;</span> Arriving<br>
        <span style="color:green">&#9644;</span> Departing<br>
        <span style="color:gray">&#9644;</span> Transiting
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html))
    m.save(output_path)
    print(f"Saved to {output_path}")
    print(f"Sample: {sum(t.is_arriving for t in sample)} arriving, "
          f"{sum(t.is_departing for t in sample)} departing, "
          f"{sum(t.is_transiting for t in sample)} transiting")

plot_tracks_classified(tracks, n=60)

Saved to figures/track_samples.html
Sample: 22 arriving, 27 departing, 20 transiting


In [9]:
Path("lfbo").mkdir(parents=True, exist_ok=True)

with open("lfbo/tracks.pkl", "wb") as f:
    pickle.dump(tracks, f)

print(f"Saved {len(tracks)} tracks to lfbo/tracks.pkl")

Saved 1332 tracks to lfbo/tracks.pkl
